# Reproducibility Pipeline

Single notebook that drives every result in the paper. Each section
calls one module of the `code/` package; intermediate JSONs land in
`results/{task}/F0/{stage}/` (per-cell core experiments) or
`results/_aux/<theme>/` (auxiliary experiments). Re-running the
notebook from a clean checkout is sufficient to regenerate the full
results tree on a single A100 (≈18 hours wall-clock at fp32 / bf16
for the 6.9B and 12B sizes).

## Outline
1. **Setup** — load model, fix seeds.
2. **Discovery** — refined-DLA → screened FV head set.
3. **Validation** — path patching → W/C/weak partition.
4. **Causal arbitration** — group W/C/joint lesion + four-condition verdict.
5. **FV-set sign-shuffle null** ($n_\text{perm}=10{,}000$).
6. **Mechanism** — QK source distribution + per-source DLA.
7. **Robustness & specificity** — split-half, cross-task, induction TOST,
   within-layer OV null, edge-level real-vs-random.
8. **Population rule-outs** — rank-1, V-cascade, V-shuffle, head-randomised.
9. **Steering** — $v_\text{FV}$ vs $v_\mathcal{W}$ vs $v_\text{PCA}$,
   per-cell logit shift and accuracy.
10. **Cross-template transfer** — vocabulary ICL (antonym, country-capital).
11. **Scale & cross-architecture extension** — Pythia 2.8B/6.9B/12B,
    Qwen2.5-{1.5,7}B, GPT-2-medium.
12. **L11.H4 case study** — V-shuffle, OV singular spectrum, V-composition.
13. **Aggregate & FWER** — collect every per-cell JSON into the
    cross-cell summary and the verdict matrix.

## 1. Setup

In [ ]:
import torch, numpy as np
from transformer_lens import HookedTransformer
from code import (config, prompts, dla, path_patching, group_lesion,
                   qk_source, per_source_dla, rule_outs, steering,
                   cross_template, scale_extension, case_study, io, utils)

utils.seed_all(0)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

In [ ]:
def load_model(spec):
    """Load a HookedTransformer matching one of the ModelSpec entries."""
    dtype = {'fp32': torch.float32, 'bf16': torch.bfloat16}[spec.dtype]
    return HookedTransformer.from_pretrained(
        spec.name, device=DEVICE, dtype=dtype, attn_implementation='eager',
    )

MAIN_CELLS = [(m, t) for m in config.PYTHIA_MAIN for t in config.RULE_TASKS]
len(MAIN_CELLS), MAIN_CELLS[0]

## 2–4. Per-cell discovery → validation → causal arbitration

Six main cells: Pythia-{410M, 1B, 1.4B} × {hierarchical, modular}.

In [ ]:
for spec, task in MAIN_CELLS:
    model = load_model(spec)
    discov = prompts.generate_batch(task.task_id, config.N_DISCOVERY,
                                       base_seed=config.DISCOVERY_SEEDS[0])
    pp_pairs = [prompts.make_paired_rule_flip(task.task_id, seed=1000+i)
                for i in range(config.N_PP)]
    eval_p = prompts.generate_batch(task.task_id, config.N_EVAL,
                                       base_seed=config.EVAL_SEED)

    # 2. refined-DLA + FV-candidate screen
    raw_dla = dla.per_head_refined_dla(model, discov)
    fv = dla.screen_fv_candidates(raw_dla)
    io.save(task.task_id, 'discover', 'refined_dla',     spec.short, {'mean_dla': raw_dla.mean(dim=0).tolist()})
    io.save(task.task_id, 'discover', 'screened_heads',  spec.short, fv)

    # 3. path patching → W/C/weak partition
    pp = path_patching.path_patch_per_head(model, pp_pairs, fv['fv_heads'])
    partition = path_patching.partition_into_W_C_weak(pp)
    io.save(task.task_id, 'validate', 'path_patching',  spec.short, pp)
    io.save(task.task_id, 'validate', 'circuit_test',   spec.short, partition)

    # 4. group W/C/joint lesion + four-condition verdict
    for strategy in ('zero', 'mean'):
        lesion = group_lesion.run(model, eval_p, partition, strategy=strategy)
        name = 'group_canceller_lesion' if strategy == 'zero' else 'group_canceller_lesion_mean'
        io.save(task.task_id, 'validate', name, spec.short, lesion)
    del model

## 5. Sign-shuffle null

In [ ]:
for spec, task in MAIN_CELLS:
    model = load_model(spec)
    eval_p = prompts.generate_batch(task.task_id, config.N_EVAL,
                                       base_seed=config.EVAL_SEED)
    partition = io.load(task.task_id, 'validate', 'circuit_test', spec.short)
    null = group_lesion.sign_shuffle_null(model, eval_p, partition,
                                            n_perm=config.SHUFFLE_NPERM)
    io.aux_save('sign_shuffle_n10k', f'sign_shuffle_{task.task_id}_{spec.short}', null)
    del model

## 6. Mechanism — QK source + per-source DLA

In [ ]:
for spec, task in MAIN_CELLS:
    model = load_model(spec)
    eval_p = prompts.generate_batch(task.task_id, config.N_EVAL,
                                       base_seed=config.EVAL_SEED)
    partition = io.load(task.task_id, 'validate', 'circuit_test', spec.short)
    qk = qk_source.run(model, eval_p, partition)
    psd = per_source_dla.run(model, eval_p, partition['cancellers'])
    io.save(task.task_id, 'mechanism', 'qk_source', spec.short, qk)
    io.aux_save('per_source_dla', f'per_source_dla_{task.task_id}_{spec.short}', psd)
    del model

## 7. Robustness & specificity

In [ ]:
# split-half, cross-task transfer, within-layer OV null and edge-level
# real-vs-random nulls are produced by the same group-lesion / partition
# pipeline on shuffled or held-out splits. See `code/group_lesion.py` and
# `code/rule_outs.py` for the exact protocols.
for spec, task in MAIN_CELLS:
    model = load_model(spec)
    eval_p = prompts.generate_batch(task.task_id, config.N_EVAL,
                                       base_seed=config.EVAL_SEED)
    partition = io.load(task.task_id, 'validate', 'circuit_test', spec.short)
    induction_top10 = io.load(task.task_id, 'controls', 'induction', spec.short)['top10']
    n_total = model.cfg.n_layers * model.cfg.n_heads
    tost = rule_outs.induction_overlap_tost(induction_top10,
                                              partition['writers'] + partition['cancellers'],
                                              n_total)
    print(f'{spec.short} {task.task_id}: induction TOST = {tost}')
    del model

## 8. Population rule-outs

In [ ]:
for spec, task in MAIN_CELLS:
    model = load_model(spec)
    eval_p = prompts.generate_batch(task.task_id, config.N_EVAL,
                                       base_seed=config.EVAL_SEED)
    partition = io.load(task.task_id, 'validate', 'circuit_test', spec.short)

    rank1 = {f'L{L}.H{H}': rule_outs.rank1_share(model, (L, H))
              for (L, H) in partition['cancellers']}
    vshuf = {f'L{L}.H{H}': rule_outs.v_shuffle(model, (L, H), eval_p)
              for (L, H) in partition['cancellers']}
    headrand = rule_outs.head_randomised_control(
        model, eval_p,
        union_size=len(partition['writers']) + len(partition['cancellers']),
        nW=len(partition['writers']), nC=len(partition['cancellers']),
    )
    io.aux_save('rank1_vcascade_per_cell',
                  f'rank1_{task.task_id}_{spec.short}', rank1)
    io.aux_save('v_shuffle_replication',
                  f'vshuffle_{task.task_id}_{spec.short}', vshuf)
    io.aux_save('head_randomized_control',
                  f'headrand_{task.task_id}_{spec.short}', headrand)
    del model

## 9. Steering

In [ ]:
# Steering-vector construction (vFV / vW / vPCA at the readout token).
# These vectors feed two downstream evaluations:
#   - held-out alpha on the 6 main cells (logit shift only; App. D.1)
#   - transplant accuracy on the 6 main cells (alpha-free; App. D.2)
# Earlier in-sample alpha per-vector evaluation has been removed as
# confounded; see paper App. D for the framing.
for spec, task in MAIN_CELLS:
    model = load_model(spec)
    discov = prompts.generate_batch(task.task_id, config.N_DISCOVERY,
                                       base_seed=config.DISCOVERY_SEEDS[0])
    partition = io.load(task.task_id, 'validate', 'circuit_test', spec.short)
    sv = steering.build_steering_vectors(model, discov, partition)
    io.aux_save('steering_vectors',
                  f'sv_{task.task_id}_{spec.short}',
                  {k: v.tolist() for k, v in sv.items() if k != 'head_means'})
    del model


## 10. Cross-template transfer

In [ ]:
src_specs = [config.PYTHIA_MAIN[0]]                 # 410M only for vocab transfer
for spec in src_specs:
    model = load_model(spec)
    for src_task in ('hierarchical', 'modular'):
        partition = io.load(src_task, 'validate', 'circuit_test', spec.short)
        for tgt in config.VOCAB_TASKS:
            r = cross_template.run(model, partition, tgt)
            io.aux_save('vocab_transfer',
                          f'group_{src_task}_to_{tgt}', r)
    solo = cross_template.l11h4_solo_per_template(
        model, ('hierarchical', 'modular') + tuple(config.VOCAB_TASKS),
    )
    io.aux_save('vocab_transfer', 'l11h4_solo_all_templates', solo)
    del model

## 11. Scale & cross-architecture extension

In [ ]:
ext_specs = list(config.PYTHIA_LADDER_EXT) + list(config.CROSS_FAMILY)
for spec in ext_specs:
    for task in config.RULE_TASKS:
        # Cross-family models are tested on modular only.
        if spec.arch in ('llama', 'conv1d') and task.task_id != 'modular':
            continue
        result = scale_extension.run_one_cell(
            lambda s=spec: load_model(s), spec.short, task.task_id,
        )
        io.aux_save('scale_extension',
                      f'cell_{task.task_id}_{spec.short}', result)

## 12. L11.H4 case study (Pythia-410M only)

In [ ]:
model = load_model(config.PYTHIA_MAIN[0])
for task in config.RULE_TASKS:
    eval_p = prompts.generate_batch(task.task_id, config.N_CASESTUDY,
                                       base_seed=config.EVAL_SEED)
    cs = case_study.run(model, eval_p)
    io.aux_save('mechinterp_l11h4',
                  f'casestudy_{task.task_id}_410m', cs)
del model

## 13. Aggregate & FWER

Once every per-cell JSON is on disk, the aggregate cell runs in seconds.
It writes the cross-cell verdict matrix, the FWER-corrected sign-shuffle
result, and the headline numbers consumed by `figures/`.

In [ ]:
from code import aggregate
aggregate.write_all()  # populates results/{task}/_aggregate/ and updates extracted_numbers.json